# How to run simulations in parallel

For fast or batched simulators, such as most synthetic ones, running the simulations in a single process is usually fine. For a simulator that takes one second per call, 10,000 simulations take almost three hours on one CPU. `sbi` can spread the simulations over several processes with `joblib`.

Pick the function by what you have:

| You have | Use |
|---|---|
| a prior (or proposal) and want training data | `simulate_for_sbi(..., num_workers=8)` |
| parameters already, e.g. from a design or another run | `simulate_from_theta(..., num_workers=8)` |
| a simulator that you call many times, e.g. in your own loop | `parallelize_simulator(simulator)` |

All three use the same parallel pipeline.

In [1]:
import time

import numpy as np
import torch

from sbi.utils import BoxUniform


def simulator(theta):
    """A slow simulator: one parameter set in, two summary statistics out."""
    time.sleep(0.05)  # Stands in for an expensive computation.
    theta = np.asarray(theta)
    trace = theta[0] + theta[1] * np.random.randn(100)
    return np.array([trace.mean(), trace.std()])


prior = BoxUniform(low=torch.zeros(2), high=torch.ones(2))

## Training data from a prior

`simulate_for_sbi` samples parameters from the prior and simulates them. Set `num_workers` to run the simulations in parallel. Unlike the other two functions, `simulate_for_sbi` assumes a batched simulator by default. Our simulator takes one parameter set per call, so we pass `simulator_is_batched=False`.

In [2]:
from sbi.inference import simulate_for_sbi

start = time.perf_counter()
theta, x = simulate_for_sbi(
    simulator,
    prior,
    num_simulations=200,
    num_workers=8,
    simulator_is_batched=False,
    seed=0,
    show_progress_bar=False,
)
print(
    f"{time.perf_counter() - start:.1f} s for {len(x)} simulations; x has shape {tuple(x.shape)}"
)

5.1 s for 200 simulations; x has shape (200, 2)


One worker would need about 10 s here (200 × 0.05 s). The first parallel call also takes a few seconds to start the workers; later calls reuse them. So a short test run shows less speedup than a real run.

The simulator receives the sampled parameters as a tensor: on the CPU if there is more than one worker, otherwise on the device of the prior. Our simulator converts them with `np.asarray`, which a simulator that uses numpy-only methods, such as `theta.astype`, needs as well. For a prior on a GPU, convert with `np.asarray(theta.cpu())`. The returned `theta` and `x` are float32 tensors, ready for `append_simulations`.

## Parameters you already have

`simulate_from_theta` simulates given parameters and returns the outputs in the same order. The simulator receives the parameters as the type of `theta`, a tensor or a numpy array, and the outputs keep the type that the simulator returns.

`simulation_batch_size` sets how many parameter sets one worker task simulates. The default, `"auto"`, splits the parameter sets into four tasks per worker: few tasks keep the cost of sending work to the workers low, and several tasks per worker balance uneven run times.

In [3]:
from sbi.utils import simulate_from_theta

theta = prior.sample((200,))
x = simulate_from_theta(simulator, theta, num_workers=8, show_progress_bar=False)
print(type(x).__name__, x.shape)

ndarray (200, 2)


Our simulator returns numpy arrays, so `x` is a numpy array. `append_simulations` needs float32 tensors, so convert it first:

In [4]:
x = torch.as_tensor(x, dtype=torch.float32)

## A parallel simulator you can reuse

`parallelize_simulator` turns your simulator into a batched simulator that runs in parallel on every call. By default, it uses all CPUs and passes one parameter set per call.

In [5]:
from sbi.utils import parallelize_simulator

parallel_simulator = parallelize_simulator(simulator, num_workers=8)

x = parallel_simulator(prior.sample((200,)))
print(x.shape)

(200, 2)


You can also use it as a decorator on the definition of your simulator:

```python
@parallelize_simulator(num_workers=8)
def simulator(theta):
    ...
```

The parallel simulator is batched. To use it in `simulate_for_sbi`, pass the number of simulations as `simulation_batch_size`, so that it receives all parameter sets in one call and spreads them over its workers. With the default on one worker, it would receive one parameter set per call and start a parallel run for each one, which is much slower than running the simulator directly.

In [6]:
theta, x = simulate_for_sbi(
    parallel_simulator,
    prior,
    num_simulations=200,
    simulation_batch_size=200,
    show_progress_bar=False,
)

## Batched simulators

If your simulator takes a batch of parameter sets per call, pass `simulator_is_batched=True`. Then `simulation_batch_size` is the number of parameter sets per call. On one worker, the default passes one parameter set per call; a vectorized simulator is faster with larger batches, so set `simulation_batch_size`, e.g. to the number of simulations. Fast vectorized simulators rarely gain from parallel workers.

The defaults differ: `simulate_for_sbi` assumes a batched simulator, `simulate_from_theta` and `parallelize_simulator` assume one parameter set per call. If a batched simulator does not return one output per parameter set, or if it raises an exception, the error suggests `simulator_is_batched=False`. The opposite mistake gets no such hint. A batched simulator passed with `simulator_is_batched=False` receives one parameter set of shape `(num_parameters,)` per call: code such as `theta[:, 0]` raises an `IndexError`, and code that sizes its outputs by `len(theta)` runs, but each output gains a dimension.

## Simulators that write files

Some simulators run an external program that writes its results to disk. Return the file path: `simulate_from_theta` and `parallelize_simulator` return non-numeric outputs as a list, in the order of `theta`. `simulate_for_sbi` needs numeric outputs, so load and summarize the files first.

In [7]:
def run_external_program(theta):
    path = f"simulation_{theta[0]:.3f}_{theta[1]:.3f}.npy"
    # Your program would write its result to `path` here.
    return path


paths = simulate_from_theta(
    run_external_program, prior.sample((3,)), num_workers=2, show_progress_bar=False
)
print(paths)

['simulation_0.355_0.611.npy', 'simulation_0.011_0.178.npy', 'simulation_0.283_0.420.npy']


## Failed simulations

By default, an exception in the simulator stops all simulations and is raised. A note on the exception names the failed parameter set.

In a long run, you may want to keep the finished simulations. With `on_error="nan"`, a failed simulation returns NaN (or `None` for outputs that are not numeric), and one warning reports how many failed. If all simulations fail, the first exception is raised.

With more than one worker, the exception comes back from a worker process. An exception that cannot be sent between processes, for example one whose class is defined in a script or notebook, arrives as a `RuntimeError` with the original type and message.

In [8]:
def fragile_simulator(theta):
    if theta[0] > 0.9:
        raise RuntimeError("The solver diverged.")
    return simulator(theta)


theta = prior.sample((200,))
x = simulate_from_theta(
    fragile_simulator,
    theta,
    num_workers=8,
    on_error="nan",
    show_progress_bar=False,
)
print(f"{np.isnan(x).any(axis=1).sum()} of {len(x)} simulations failed")

26 of 200 simulations failed


  x = simulate_from_theta(


Failures often cluster in one region of the parameter space, as here for `theta[0] > 0.9`. If you discard the failed simulations, the posterior describes only the parameters that do not fail. See the FAQ entry [What should I do when my simulator returns NaN?](../faq/question_02_nans.md) for how `sbi` handles NaN in the training data.

## Reproducible simulations

`simulate_for_sbi` and `simulate_from_theta` take a `seed`. `simulate_from_theta` reseeds the simulator before each call: before each simulation, or before each batch for a batched simulator. For a simulator that takes one parameter set per call, the outputs then depend on neither `num_workers` nor `simulation_batch_size`. For a batched simulator, they do not depend on `num_workers` if you fix `simulation_batch_size`, because the default, `"auto"`, changes with `num_workers`. `simulate_from_theta` restores your random state afterwards.

`simulate_for_sbi` seeds once, before it samples the parameters, and leaves the random state seeded. It reseeds a simulator that takes one parameter set per call before each simulation, as `simulate_from_theta` does. A batched simulator continues from the seeded state on one worker and gets a seed per batch with more workers, so its outputs change with `num_workers`.

With more than one worker, the simulator receives tensors on the CPU. If `theta` is on a GPU and your simulator draws noise on the device of `theta`, the noise changes with `num_workers` too.

Seeding needs joblib's default workers, which are separate processes. If you configure joblib to use threads instead, the threads share one random state, so a `seed` raises an error.

In [9]:
x1 = simulate_from_theta(
    simulator, theta, num_workers=8, seed=1, show_progress_bar=False
)
x2 = simulate_from_theta(
    simulator,
    theta,
    num_workers=2,
    simulation_batch_size=7,
    seed=1,
    show_progress_bar=False,
)
print("same seed, same simulations:", np.array_equal(x1, x2))

same seed, same simulations: True


`parallelize_simulator` takes no seed, so that repeated calls draw new noise. To reproduce a run, seed once before it:

In [10]:
from sbi.utils.sbiutils import seed_all_backends

seed_all_backends(2)
first_run = parallel_simulator(theta), parallel_simulator(theta)
seed_all_backends(2)
second_run = parallel_simulator(theta), parallel_simulator(theta)
print("runs reproduce:", all(map(np.array_equal, first_run, second_run)))
print("calls differ:", not np.array_equal(*first_run))

runs reproduce: True
calls differ: True


## Random numbers in your simulator

`sbi` seeds the global random state of numpy, torch and Python's `random` module in every worker. Draw your noise from these, e.g. with `np.random.randn` or `torch.randn`.

A numpy `Generator` that you create outside the simulator is different, and `seed` does not reproduce its noise. With one worker, the simulator uses your `Generator` itself. With more workers, every worker task gets a copy of it in the same state, so the tasks repeat each other's noise:

In [11]:
rng = np.random.default_rng(0)


def simulator_with_shared_rng(theta):
    return np.asarray(theta) + rng.normal(size=2)


x = simulate_from_theta(
    simulator_with_shared_rng, np.zeros((6, 2)), num_workers=2, show_progress_bar=False
)
print("distinct simulations:", len(np.unique(x, axis=0)))

distinct simulations: 1


Create the `Generator` inside the simulator, from the seeded global state:

In [12]:
def simulator_with_own_rng(theta):
    rng = np.random.default_rng(np.random.randint(2**32 - 1, dtype=np.int64))
    return np.asarray(theta) + rng.normal(size=2)


x = simulate_from_theta(
    simulator_with_own_rng, np.zeros((6, 2)), num_workers=2, show_progress_bar=False
)
print("distinct simulations:", len(np.unique(x, axis=0)))

distinct simulations: 6


## Troubleshooting

- **Pickling errors:** see the FAQ entry [When using multiple workers, I get a pickling error](../faq/question_03_pickling_error.md).
- **The program hangs, or simulations start over and over:** if your simulator lives in a module, do not start simulations when that module is imported. The workers import the module, which can deadlock or start the simulations again.
- **Every call is slow:** each call sends the simulator to the workers, together with the module-level objects it reads, unless the workers can import it. A simulator defined in a script or notebook is always sent, and so is a decorated one. If it reads large objects, such as a lookup table, define it in a module file and use the function form, `parallelize_simulator(simulator)`.